# Practical 07 — Finance Research-Brief Agent

Chapter 7 ("Other Applications and Future Trends") frames many emerging finance
applications — research assistants, disclosure monitors, automated analyst notes — as a
single **route -> gather -> write** loop: send a question to the right evidence source,
collect *grounded* evidence, then write a short cited brief. The agentic version of this
project runs in Claude Code / Cline, where the model only chooses inputs and interprets
outputs while deterministic tools do all routing, retrieval, and figure lookup.

This notebook is the Colab-friendly counterpart: it drives those same reference tools
(`tools/route.py`, `tools/metrics.py`, `tools/retrieve.py`) directly, end to end, fully
offline, over the bundled fictional company **Meridian Robotics Inc.**, so you can watch
the whole pipeline run and see its real outputs.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "07-applications-future"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## The bundled evidence

Everything is offline fictional material for **Meridian Robotics Inc.**. There are three
kinds of evidence, each backed by a different tool:

- **metrics** — a small CSV table of figures (`data/metrics.csv`), one row per figure with
  a value, unit, period, and the source document it came from.
- **filings** — 10-K disclosure text under `data/filings/` (risk factors and MD&A).
- **news** — news-wire snippets under `data/news/` (a partnership, a recall).

Let's list what we have to work with.

In [ ]:
import json
from tools._common import load_snippets, DATA_DIR
from tools.metrics import load_table

table = load_table()
print("Metrics table keys (", len(table), "rows ):")
print(json.dumps(sorted(table), indent=2))

snippets = load_snippets("all")
print("\nSnippet documents:")
for sid, text in snippets.items():
    print(f"  {sid:30s} {len(text):4d} chars")

## Step 1 — Route the question

`tools.route.classify` decides whether a question is a **metrics** lookup (a figure), a
**filings** read (steady-state disclosure), or a **news** read (a recent event). It uses
plain keyword/substring rules over a small finance vocabulary, so the choice is fully
reproducible and inspectable — no model judgement involved. The returned dict exposes the
per-route `scores`, the exact `matched` terms, and whether the phrasing is `value_seeking`
(asks for a number).

Naming a metric *and* asking for its value (e.g. "what was ... gross margin") is the
strongest signal and gets a +2 boost toward `metrics`.

In [ ]:
from tools.route import classify

questions = [
    "What was Meridian's gross margin in fiscal 2025?",   # -> metrics
    "What supply chain risks does Meridian disclose?",    # -> filings
    "What partnership did Meridian recently announce?",   # -> news
]
for q in questions:
    r = classify(q)
    print(f"Q: {q}")
    print(f"   route={r['route']!r}  value_seeking={r['value_seeking']}  scores={r['scores']}")
    print(f"   matched={ {k: v for k, v in r['matched'].items() if v} }")
    print()

## Step 2a — Gather a figure (metrics route)

When the route is `metrics`, the researcher calls `tools.metrics.lookup`. It resolves the
free-text metric name to a canonical key (longer phrases win, so "gross margin" beats bare
"margin"), reads the CSV, and returns the figure **with its `source`** so the brief can cite
it. The agent never recites a number from memory; it quotes exactly what comes back.

In [ ]:
from tools.metrics import lookup, resolve_key

print("resolve_key('gross margin')        ->", resolve_key("gross margin"))
print("resolve_key('total sales for year')->", resolve_key("total sales for the year"))
print()
print("lookup('gross margin'):")
print(json.dumps(lookup("gross margin"), indent=2))

### Try it / interpret: a figure that isn't in the table

The README suggests asking for something the table doesn't have — e.g. a **dividend yield**.
The metrics tool returns `found: false`, which is the agent's signal to say
*"Not answerable from the available sources"* rather than guess. This guardrail — refusing
to fabricate a number — is the whole point of routing figures through a deterministic tool.

In [ ]:
print("lookup('dividend yield'):")
print(json.dumps(lookup("dividend yield"), indent=2))

## Step 2b — Gather evidence (filings / news routes)

When the route is `filings` or `news`, the researcher uses `tools.retrieve`: a tiny,
deterministic TF-IDF cosine retriever (stdlib + NumPy) built over the snippets. `--source`
restricts the search to the route the router chose, and each hit is identified by its file
name so the brief can cite the exact document.

In [ ]:
from tools.retrieve import build_retriever

print("News query: 'autonomous warehouse partnership with a logistics operator'")
for sid, text, score in build_retriever("news").search(
        "autonomous warehouse partnership with a logistics operator", k=2):
    print(f"  {score:.4f}  {sid}")
    print(f"          {text[:90].strip()}...")

print("\nFilings query: 'customer concentration and supply chain risk'")
for sid, text, score in build_retriever("filings").search(
        "customer concentration and supply chain risk", k=2):
    print(f"  {score:.4f}  {sid}")
    print(f"          {text[:90].strip()}...")

The on-topic snippet ranks first in each case, and the source filter keeps news queries
from ever returning a filing (and vice versa) — exactly what the tests assert.

## Step 3 — Write a grounded, cited brief

The brief-writer reads only the gathered evidence and writes a short answer, citing a source
for every claim (a metric's `source` field, or a retrieved snippet id). Below we tie the
whole loop together in one helper: route -> gather -> assemble a brief, with a citation on
every line. This mirrors what the `/brief` command produces and saves to `reports/`.

In [ ]:
def research_brief(question, k=2):
    r = classify(question)
    route = r["route"]
    lines = [f"# Brief: {question}", "", f"_Route: **{route}** (scores {r['scores']})_", ""]
    if route == "metrics":
        hit = lookup(question)
        if hit.get("found"):
            lines.append(
                f"- {hit['metric'].replace('_', ' ')} was **{hit['value']} {hit['unit']}** "
                f"in {hit['period']}.  [source: {hit['source']}]"
            )
        else:
            lines.append("- Not answerable from the available sources "
                         f"(no metric matched '{question}').")
    else:
        hits = build_retriever(route).search(question, k=k)
        on_topic = [(sid, text, s) for sid, text, s in hits if s > 0.0]
        if not on_topic:
            lines.append("- Not answerable from the available sources (no on-topic snippet).")
        else:
            for sid, text, s in on_topic:
                first = text.strip().splitlines()
                quote = next((ln for ln in first[1:] if ln.strip()), first[0]).strip()
                lines.append(f"- {quote[:160]}...  [source: {sid}, score {s:.3f}]")
    return "\n".join(lines)

for q in [
    "What was Meridian's gross margin in fiscal 2025?",
    "What partnership did Meridian recently announce?",
    "What is Meridian's dividend yield?",
]:
    print(research_brief(q))
    print("\n" + "-" * 72 + "\n")

### Try it / interpret: a two-part question routes once, then needs two gathers

The README highlights questions that ask for a figure *and* the reason behind it — e.g.
"What was gross margin, and why did it improve?". The keyword router lands on a single
route, but a good agent recognises it needs **two** gathers: a `metrics` lookup for the
number and a `filings` retrieval for the explanation, citing both. Below we inspect the
routing scores and then perform both gathers explicitly.

In [ ]:
twopart = "What was gross margin, and why did it improve?"
print("Routing scores:", classify(twopart)["scores"], "->", classify(twopart)["route"])
print()
print("Figure  ->", json.dumps(lookup("gross margin")))
print()
print("Reason  -> filings retrieval for 'gross margin improvement software services':")
for sid, text, score in build_retriever("filings").search(
        "gross margin improvement software service contracts", k=1):
    reason = next((ln for ln in text.splitlines()[1:] if "margin" in ln.lower()),
                  text.splitlines()[0])
    print(f"  [{sid}, score {score:.3f}] {reason.strip()[:150]}...")

## Recap / next steps

We ran the Chapter 7 research-brief pipeline end to end on the real bundled data, entirely
offline, by driving the reference tools directly:

1. **Route** — `tools.route.classify` picks `metrics` / `filings` / `news` from inspectable
   keyword scores.
2. **Gather** — `tools.metrics.lookup` returns a cited figure (or `found: false`);
   `tools.retrieve` does deterministic TF-IDF retrieval over the snippets, filtered to the
   chosen source.
3. **Write** — assemble a short brief with a citation on every line, refusing to answer when
   the evidence isn't there.

**Things to try next** (from the README): add a new row to `data/metrics.csv` plus a matching
alias in `tools/metrics.py` and ask for it; add a fourth snippet under `data/news/` and ask a
question only it can answer; or phrase a metric question as discussion ("Tell me about the
company's risks") and read `classify(...)['matched']` to see exactly why it routed where it
did. To experience the full agentic loop — including the LLM choosing inputs and writing the
prose — open this folder in Claude Code or Cline and run `/brief "<your question>"`.